<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [48]:
from torch import randn as torch_randn
from fastai.vision.all import test_eq

To do: change code to use only monai resnet

In [50]:
from torch import device as torch_device
from torch.cuda import is_available as cuda_is_available
device = torch_device("cuda" if cuda_is_available() else "cpu")
print(device)

cuda


## DnCNN

In [1]:
#| echo: false
#| output: asis
show_doc(DnCNN)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/nets.py#L23){target="_blank" style="float:right; font-size:smaller"}

### DnCNN

>      DnCNN (channels, num_of_layers=9, features=64, kernel_size=3)

Base class for all neural network modules.

Your models should also subclass this class.

Modules can also contain other Modules, allowing to nest them in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will have their
parameters converted too when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [52]:
x = torch_randn(16, 1, 32, 64)

tst = DnCNN(1)
test_eq(tst(x).shape, x.shape)

## UNet

### Convolutional sub-unit

In [2]:
#| echo: false
#| output: asis
show_doc(SubNetConv)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/nets.py#L43){target="_blank" style="float:right; font-size:smaller"}

### SubNetConv

>      SubNetConv (ks=3, stride=1, padding=None, bias=None, ndim=2,
>                  norm_type=<NormType.Batch: 1>, bn_1st=True, act_cls=<class
>                  'torch.nn.modules.activation.ReLU'>, transpose=False,
>                  init='auto', xtra=None, bias_std=0.01, dropout=0.0)

In [54]:
x = torch_randn(16, 1, 32, 64, 64)
xdim = len(x.shape)-2

# reduce
tst = SubNetConv(3, padding=1, stride=2, ndim=xdim,
                 norm_type=NormType.Batch, dropout=.1)(1, 2, 2)
y = tst(x)
test_eq(y.shape, [16, 2, 8, 16, 16])
print(tst)
# upsample
tst = SubNetConv(ks=4, padding=0, stride=4, ndim=xdim, norm_type=NormType.Batch,
                 transpose=True)(2, 1)  # to double the size, the kernel cannot be odd
test_eq(tst(y).shape, x.shape)
del y

Sequential(
  (0): Sequential(
    (0): ConvLayer(
      (0): Conv3d(1, 2, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
      (1): BatchNorm3d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (1): Dropout(p=0.1, inplace=False)
  )
  (1): Sequential(
    (0): ConvLayer(
      (0): Conv3d(2, 2, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
      (1): BatchNorm3d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (1): Dropout(p=0.1, inplace=False)
  )
)


### UNet block

### Recursive UNet

In [3]:
#| echo: false
#| output: asis
show_doc(UNet)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/nets.py#L135){target="_blank" style="float:right; font-size:smaller"}

### UNet

>      UNet (depth=4, mult_chan=32, in_channels=1, out_channels=1,
>            last_activation=None, kernel_size=3, ndim=2, n_conv_per_depth=2,
>            activation='ReLU', norm_type=<NormType.Batch: 1>, dropout=0.0,
>            pool=<function MaxPool>, pool_size=2, residual=False,
>            prob_out=False, eps_scale=0.001)

Base class for all neural network modules.

Your models should also subclass this class.

Modules can also contain other Modules, allowing to nest them in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will have their
parameters converted too when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| depth | int | 4 | depth of the UNet network |
| mult_chan | int | 32 | number of filters at first layer |
| in_channels | int | 1 | number of input channels |
| out_channels | int | 1 | number of output channels |
| last_activation | NoneType | None | last activation before final result |
| kernel_size | int | 3 | kernel size of convolutional layers |
| ndim | int | 2 | number of spatial dimensions of the input data |
| n_conv_per_depth | int | 2 | number of convolutions per layer |
| activation | str | ReLU | activation function used in convolutional layers |
| norm_type | NormType | NormType.Batch |  |
| dropout | float | 0.0 |  |
| pool | function | MaxPool |  |
| pool_size | int | 2 |  |
| residual | bool | False |  |
| prob_out | bool | False |  |
| eps_scale | float | 0.001 |  |

In [57]:
x = torch_randn(16, 1, 32, 64, 64)
xdim = len(x.shape)-2

tst = UNet(depth=1, ndim=xdim, n_conv_per_depth=1, residual=True)
mods = list(tst.children())
print(mods)
test_eq(tst(x).shape, [16, 1, 32, 64, 64])

[_Net_recurse(
  (sub_conv_more): ConvLayer(
    (0): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (sub_u): Sequential(
    (0): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (1): _Net_recurse(
      (sub_conv_more): ConvLayer(
        (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
    )
    (2): Upsample(scale_factor=2.0, mode='nearest')
  )
  (sub_conv_less): ConvLayer(
    (0): Conv3d(96, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
), ConvLayer(
  (0): Conv3d(32, 1, kernel_size=(3, 3, 3), stride=(1, 

## DeepLab v3+

### Config

In [58]:
# @dataclass
# class DeeplabConfig:
#     dimensions: int
#     in_channels: int
#     out_channels: int
#     backbone: str = "xception" 
#     pretrained: bool = False
#     middle_flow_blocks: int = 16
#     aspp_dilations: List[int] = field(default_factory=lambda: [1, 6, 12, 18])
#     entry_block3_stride: int = 2
#     middle_block_dilation: int = 1
#     exit_block_dilations: Tuple[int, int] = (1, 2)

# def get_padding(kernel_size: int, dilation: int) -> int:
#     return (kernel_size - 1) * dilation // 2
    
# def interpolate(x: torch_Tensor, size: Union[List[int], Tuple[int, ...]], dims: int) -> torch_Tensor:
#     if dims == 2:
#         return F.interpolate(x, size=size, mode='bilinear', align_corners=True)
#     elif dims == 3:
#         return F.interpolate(x, size=size, mode='trilinear', align_corners=True)
#     else:
#         raise ValueError(f"Unsupported number of dimensions: {dims}")

### Blocks

In [59]:
# class SeparableConv(nn.Module):
#     def __init__(self, config: DeeplabConfig, inplanes: int, planes: int, kernel_size: int = 3, stride: int = 1, 
#                  dilation: int = 1, bias: bool = False, norm: Optional[str] = None):
#         super().__init__()
#         self.conv1 = Convolution(
#             spatial_dims=config.dimensions, 
#             in_channels=inplanes, 
#             out_channels=inplanes, 
#             kernel_size=kernel_size,
#             groups=inplanes, 
#             padding=get_padding(kernel_size, dilation), 
#             dilation=dilation, 
#             bias=bias, 
#             strides=stride
#         )
#         self.pointwise = Convolution(
#             spatial_dims=config.dimensions, 
#             in_channels=inplanes, 
#             out_channels=planes, 
#             kernel_size=1, 
#             strides=1,
#             padding=0, 
#             dilation=1, 
#             groups=1, 
#             bias=bias,
#             norm=Norm.BATCH if norm else None
#         )

#     def forward(self, x: torch_Tensor) -> torch_Tensor:
#         x = self.conv1(x)
#         x = self.pointwise(x)
#         return x

# class Block(nn.Module):
#     def __init__(self, config: DeeplabConfig, inplanes: int, planes: int, reps: int, stride: int = 1, 
#                  dilation: int = 1, start_with_relu: bool = True, grow_first: bool = True, 
#                  is_last: bool = False):
#         super().__init__()
#         if planes != inplanes or stride != 1:
#             self.skip = Convolution(config.dimensions, inplanes, planes, kernel_size=1, bias=False, 
#                                     strides=stride, norm=Norm.BATCH)
#         else:
#             self.skip = None

#         self.relu = nn.ReLU(inplace=True)
#         rep = []

#         filters = inplanes
#         if grow_first:
#             rep.append(self.relu)
#             rep.append(SeparableConv(config, inplanes, planes, 3, stride=1, dilation=dilation, norm=Norm.BATCH))
#             filters = planes

#         for _ in range(reps - 1):
#             rep.append(self.relu)
#             rep.append(SeparableConv(config, filters, filters, 3, stride=1, dilation=dilation, norm=Norm.BATCH))

#         if not grow_first:
#             rep.append(self.relu)
#             rep.append(SeparableConv(config, inplanes, planes, 3, stride=1, dilation=dilation, norm=Norm.BATCH))

#         if not start_with_relu:
#             rep = rep[1:]

#         if stride != 1:
#             rep.append(SeparableConv(config, planes, planes, 3, stride=2))

#         if stride == 1 and is_last:
#             rep.append(SeparableConv(config, planes, planes, 3, stride=1))

#         self.rep = nn.Sequential(*rep)

#     def forward(self, inp: torch_Tensor) -> torch_Tensor:
#         x = self.rep(inp)
#         if self.skip is not None:
#             skip = self.skip(inp)
#         else:
#             skip = inp
#         x += skip
#         return x

### Aligned Xception

In [60]:
# class Xception(nn.Module):
#     def __init__(self, config: DeeplabConfig):
#         super().__init__()
#         self.config = config

#         self.conv1 = Convolution(config.dimensions, config.in_channels, 32, kernel_size=3,
#                                  bias=False, strides=2, padding=1, norm=Norm.BATCH)
#         self.relu = nn.ReLU(inplace=True)
#         self.conv2 = Convolution(config.dimensions, 32, 64, kernel_size=3,
#                                  bias=False, strides=1, padding=1, norm=Norm.BATCH)

#         self.block1 = Block(config, 64, 128, reps=2, stride=2, start_with_relu=False)
#         self.block2 = Block(config, 128, 256, reps=2, stride=2, start_with_relu=True, grow_first=True)
#         self.block3 = Block(config, 256, 728, reps=2, stride=config.entry_block3_stride, 
#                             start_with_relu=True, grow_first=True, is_last=True)

#         # Middle flow
#         self.middle_flow = nn.Sequential(*[
#             Block(config, 728, 728, reps=3, stride=1, dilation=config.middle_block_dilation,
#                   start_with_relu=True, grow_first=True)
#             for _ in range(config.middle_flow_blocks)
#         ])

#         # Exit flow
#         self.exit_block = Block(config, 728, 1024, reps=2, stride=1, dilation=config.exit_block_dilations[0],
#                              start_with_relu=True, grow_first=False, is_last=True)

#         self.conv3 = SeparableConv(config, 1024, 1536, 3, stride=1, dilation=config.exit_block_dilations[1], norm=Norm.BATCH)
#         self.conv4 = SeparableConv(config, 1536, 1536, 3, stride=1, dilation=config.exit_block_dilations[1], norm=Norm.BATCH)
#         self.conv5 = SeparableConv(config, 1536, 2048, 3, stride=1, dilation=config.exit_block_dilations[1], norm=Norm.BATCH)

#     def forward(self, x: torch_Tensor) -> Tuple[torch_Tensor, torch_Tensor]:
#         # Entry flow
#         x = self.conv1(x)
#         x = self.relu(x)
#         x = self.conv2(x)
#         x = self.relu(x)

#         x = self.block1(x)
#         low_level_feat = x
#         x = self.block2(x)
#         x = self.block3(x)

#         # Middle flow
#         x = self.middle_flow(x)

#         # Exit flow
#         x = self.exit_block(x)
#         x = self.conv3(x)
#         x = self.relu(x)
#         x = self.conv4(x)
#         x = self.relu(x)
#         x = self.conv5(x)
#         x = self.relu(x)

#         return x, low_level_feat

### ASPP

In [61]:
# class ASPP_module(nn.Module):
#     def __init__(self, config: DeeplabConfig, inplanes: int, planes: int, dilation: int):
#         super().__init__()
#         kernel_size = 1 if dilation == 1 else 3
#         padding = 0 if dilation == 1 else dilation
#         self.atrous_convolution = Convolution(config.dimensions, inplanes, planes, kernel_size=kernel_size,
#                                               strides=1, padding=padding, dilation=dilation, 
#                                               bias=False, norm=Norm.BATCH)
#         self.relu = nn.ReLU()

#     def forward(self, x: torch_Tensor) -> torch_Tensor:
#         x = self.atrous_convolution(x)
#         return self.relu(x)

### DeepLab

In [62]:
# class Deeplab(nn.Module):
#     def __init__(self, config: DeeplabConfig):
#         super().__init__()
#         self.config = config

#         # Choose backbone based on configuration
#         if config.backbone == "xception":
#             self.backbone = Xception(config)
#             backbone_out_channels = 2048
#             low_level_channels = 128  # This might need adjustment based on Xception implementation
#         elif config.backbone == "resnet50":
#             if config.dimensions == 2:
#                 self.backbone = resnet50(pretrained=True)
#                 del self.backbone.fc
#                 del self.backbone.avgpool
#             else:
#                 self.backbone = resnet.resnet50(pretrained=False, spatial_dims=3)
#                 del self.backbone.fc
#                 del self.backbone.avgpool
#             backbone_out_channels = 2048
#             low_level_channels = 256  # ResNet50's first layer output
#         elif config.backbone == "resnet101":
#             if config.dimensions == 2:
#                 self.backbone = resnet101(pretrained=True)
#                 del self.backbone.fc
#                 del self.backbone.avgpool
#             else:
#                 self.backbone = resnet.resnet101(pretrained=False, spatial_dims=3)
#                 del self.backbone.fc
#                 del self.backbone.avgpool
#             backbone_out_channels = 2048
#             low_level_channels = 256  # ResNet101's first layer output
#         else:
#             raise ValueError(f"Unsupported backbone: {config.backbone}")

#         # ASPP
#         self.aspp_modules = nn.ModuleList([
#             ASPP_module(config, backbone_out_channels, 256, dilation=dilation)
#             for dilation in config.aspp_dilations
#         ])

#         self.global_avg_pool = nn.Sequential(
#             Pool[Pool.ADAPTIVEAVG, config.dimensions](1),
#             Convolution(config.dimensions, backbone_out_channels, 256, kernel_size=1, strides=1, bias=False, norm=Norm.BATCH),
#             nn.ReLU()
#         )

#         self.conv1 = Convolution(config.dimensions, 1280, 256, kernel_size=1, bias=False, norm=Norm.BATCH)
#         self.conv2 = Convolution(config.dimensions, low_level_channels, 48, kernel_size=1, bias=False, norm=Norm.BATCH)

#         self.last_conv = nn.Sequential(
#             Convolution(config.dimensions, 304, 256, kernel_size=3, strides=1, padding=1, bias=False, norm=Norm.BATCH),
#             nn.ReLU(),
#             Convolution(config.dimensions, 256, 256, kernel_size=3, strides=1, padding=1, bias=False, norm=Norm.BATCH),
#             nn.ReLU(),
#             Convolution(config.dimensions, 256, config.out_channels, kernel_size=1, strides=1)
#         )

#     def forward(self, input: torch_Tensor) -> torch_Tensor:
#         if self.config.backbone == "xception":
#             x, low_level_features = self.backbone(input)
#         else:
#             x = self.backbone.conv1(input)
#             x = self.backbone.bn1(x)
#             x = self.backbone.relu(x)
#             x = self.backbone.maxpool(x)

#             low_level_features = self.backbone.layer1(x)
#             x = self.backbone.layer2(low_level_features)
#             x = self.backbone.layer3(x)
#             x = self.backbone.layer4(x)

#         aspp_results = [module(x) for module in self.aspp_modules]
#         x5 = self.global_avg_pool(x)
#         x5 = interpolate(x5, size=x.shape[2:], dims=self.config.dimensions)
#         x = torch_cat(aspp_results + [x5], dim=1)

#         x = self.conv1(x)
#         x = interpolate(x, size=low_level_features.shape[2:], dims=self.config.dimensions)

#         low_level_features = self.conv2(low_level_features)

#         x = torch_cat((x, low_level_features), dim=1)
#         x = self.last_conv(x)
#         x = interpolate(x, size=input.shape[2:], dims=self.config.dimensions)

#         return x

#### Example

In [63]:
# # For 2D images
# config_2d = DeeplabConfig(
#     dimensions=2,
#     in_channels=3,  # For RGB images
#     out_channels=4,
#     backbone="xception",  # or whatever backbone you're using
#     aspp_dilations=[1, 6, 12, 18]
# )
# model_2d = Deeplab(config_2d)

# # For 3D images
# config_3d = DeeplabConfig(
#     dimensions=3,
#     in_channels=1,  # For single-channel 3D medical images
#     out_channels=4,
#     middle_flow_blocks=16,
#     aspp_dilations=[1, 6, 12, 18]
# )
# model_3d = Deeplab(config_3d)

In [64]:
# x = torch_randn(16,3, 64, 64)
# y = model_2d(x)

In [65]:
# from monai.utils import set_determinism
# from torch import no_grad as torch_no_grad

In [66]:
# def test_deeplab(config, input_shape, expected_output_shape):
#     set_determinism(0)  # For reproducibility
    
#     model = Deeplab(config)
#     model.eval()  # Set the model to evaluation mode
    
#     # Generate random input tensor
#     x = torch_randn(*input_shape)
    
#     # Forward pass
#     with torch_no_grad():
#         output = model(x)
    
#     # Check output shape
#     assert output.shape == expected_output_shape, f"Expected shape {expected_output_shape}, but got {output.shape}"
    
#     print(f"Test passed for {config.dimensions}D model with backbone {config.backbone}")
#     print(f"Input shape: {input_shape}")
#     print(f"Output shape: {output.shape}")
#     print("---")

# # Test 2D model
# config_2d = DeeplabConfig(
#     dimensions=2,
#     in_channels=3,
#     out_channels=4,
#     backbone="xception",
#     aspp_dilations=[1, 6, 12, 18]
# )
# test_deeplab(config_2d, (1, 3, 64, 64), (1, 4, 64, 64))

# # Test 2D model with ResNet50 backbone
# config_2d_resnet = DeeplabConfig(
#     dimensions=2,
#     in_channels=3,
#     out_channels=4,
#     backbone="resnet50",
#     aspp_dilations=[1, 6, 12, 18]
# )
# test_deeplab(config_2d_resnet, (1, 3, 64, 64), (1, 4, 64, 64))

# # Test 3D model
# config_3d = DeeplabConfig(
#     dimensions=3,
#     in_channels=1,
#     out_channels=4,
#     backbone="xception",
#     aspp_dilations=[1, 6, 12, 18]
# )
# test_deeplab(config_3d, (1, 1, 64, 64, 64), (1, 4, 64, 64, 64))

# # Test 3D model with ResNet50 backbone
# config_3d_resnet = DeeplabConfig(
#     dimensions=3,
#     in_channels=1,
#     out_channels=4,
#     backbone="resnet50",
#     aspp_dilations=[1, 6, 12, 18]
# )
# # test_deeplab(config_3d_resnet, (1, 1, 64, 64, 64), (1, 4, 64, 64, 64))

# print("All tests passed successfully!")

---